# GraphSAGE on Amazon Computers Dataset

Node classification using GraphSAGE with full-batch training.

**Dataset:** Amazon Computers (13,752 nodes, 491,722 edges, 767 features, 10 classes)

## GraphSAGE Baseline

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv
from torch.nn import BatchNorm1d

# 1. Load Amazon Computers Dataset and Generate Masks
# Amazon Computers doesn't have default masks, so we split it: 70% train, 10% val, 20% test
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

print(f'Dataset: {dataset}')
print(f'Nodes: {data.num_nodes}, Edges: {data.num_edges}')
print(f'Features: {dataset.num_node_features}, Classes: {dataset.num_classes}')
print(f'Train: {data.train_mask.sum().item()}, Val: {data.val_mask.sum().item()}, Test: {data.test_mask.sum().item()}\n')

class UpgradedGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # 1. Use SAGEConv for neighborhood aggregation
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        # 2. Add Batch Normalization to stabilize training
        self.bn1 = BatchNorm1d(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x) # Apply normalization before activation
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Setup - Amazon Computers is small enough for full-batch training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UpgradedGNN(dataset.num_node_features, 256, dataset.num_classes).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# 4. Define Training and Testing Functions (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execute the Training Loop
print(f"Starting GraphSAGE Baseline training on {device}...")
for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}')

print(f'\nTraining Complete!')
print(f'Final Test F1uracy: {test_f1:.4f}')

## GraphSAGE with MLP head

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch_geometric.datasets import Amazon
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv, BatchNorm

# 1. Load Amazon Computers Dataset
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Amazon(root='./data/Amazon', name='Computers', transform=transform)
data = dataset[0]

# 2. GraphSAGE Architecture with MLP Head
class GraphSAGEWithMLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        # --- GNN Encoder (Feature Aggregation) ---
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)

        # --- MLP Task Head (Node-wise classification logic) ---
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        # Encoder Phase
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        
        # Task Head Phase
        x = self.mlp(x)
        return x

# 3. Initialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphSAGEWithMLP(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=dataset.num_classes
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# 4. Training and Testing Functions (Full-Batch)
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    pred_cpu = pred.cpu().numpy()
    y_cpu = data.y.cpu().numpy()
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        m = mask.cpu().numpy()
        accs.append(f1_score(y_cpu[m], pred_cpu[m], average='macro'))
    return accs

# 5. Execution Loop
print(f"Training GraphSAGE + MLP Head on {device}...")
best_val_f1 = 0.0

for epoch in range(1, 201):
    loss = train()
    train_f1, val_f1, test_f1 = test()
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'Amazon_graphSAGE_MLP_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Val F1: {val_f1:.4f}, Test F1: {test_f1:.4f}')

print(f"\nTraining Complete; Best Validation Macro-F1: {best_val_f1:.4f}")